# Specialiserede Modeller — 6: Autoencodere — kunsten at glemme det uvæsentlige

Dagens model er den mærkeligste, I har mødt: et netværk, der trænes til at
**genskabe sit eget input**. "Gæt hvad du lige har fået at se"?! Det lyder som verdens
nemmeste opgave — indtil man ser fidusen: midt i netværket sidder en **flaskehals** så
snæver, at billedet IKKE kan komme helskindet igennem. Netværket TVINGES til at
komprimere: gemme det væsentlige, smide resten væk.

Det giver tre superkræfter: **kompression**, **støjfjernelse** og
**anomali-detektion** — og som sidegevinst et smukt indblik i, hvad et netværk
egentlig "forstår". Data: jeres gamle venner, de håndskrevne MNIST-cifre (kendt stof,
så al opmærksomheden kan gå til den nye modeltype).

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter MNIST (nedskaleret) fra GitHub (Plan B: upload filerne via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_test_lille.csv.gz

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/hjaelpefunktioner.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from hjaelpefunktioner import vis_billeder, vis_rekonstruktioner

torch.manual_seed(42)
np.random.seed(42)

traen_df = pd.read_csv("mnist_traen_lille.csv.gz").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv("mnist_test_lille.csv.gz").reset_index(drop=True)
print("træning:", traen_df.shape, "| test:", test_df.shape)

> **Plan B:** Fejler Kaggle, ligger der mindre udgaver i Intro-ML-mappen på GitHub —
> fjern `#`'erne (og spring `sample` over; filerne er allerede skåret ned):

In [ ]:
#  Plan B — kør KUN hvis Kaggle-cellen fejlede:
# traen_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_traen_lille.csv.gz")
# test_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_test_lille.csv.gz")
# print(traen_df.shape, test_df.shape)

In [ ]:
X_traen = torch.tensor(traen_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_traen = torch.tensor(traen_df["label"].values, dtype=torch.long)   # labels bruges KUN til at farve plots!
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print(X_traen.shape, "— flade billeder som i Intro-ML")

# 10: Flaskehalsen — 784 tal ind, 32 tal i midten, 784 tal ud

En autoencoder består af to halvdele:

- **Encoderen** presser billedet ned: 784 pixels →... → fx **32 tal** (flaskehalsen —
 også kaldet den *latente kode*)
- **Decoderen** puster det op igen: 32 tal →... → 784 pixels

Og tabsfunktionen? Bare MSE mellem input og output: "hvor tæt kom du på originalen?".
Bemærk det geniale: **der bruges INGEN labels** — billedet er sit eget facit. Det er
unsupervised learning, ligesom clustering, bare med neurale netværk.

Hvorfor lærer flaskehalsen noget klogt? Fordi 32 tal er ALT for lidt til 784 pixels —
netværket kan ikke gemme pixel-værdierne råt. Det tvinges til at opdage, at håndskrevne
cifre slet ikke er vilkårlige pixelmønstre, men varianter over ~10 former med
buer, streger og hældninger — og DET kan beskrives med få tal:

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, flaskehals=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, flaskehals))            # ned til fx 32 tal
        self.decoder = nn.Sequential(
            nn.Linear(flaskehals, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid())     # sigmoid: pixels skal ligge i [0, 1]

    def forward(self, x):
        kode = self.encoder(x)
        return self.decoder(kode)

model = Autoencoder(flaskehals=32)
print(model)

In [ ]:
torch.manual_seed(42)
model = Autoencoder(flaskehals=32)
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoke in range(10):
    for i in range(0, len(X_traen), 64):
        X_batch = X_traen[i:i + 64]
        optimizer.zero_grad()
        tab = tabsfunktion(model(X_batch), X_batch)    # facit = inputtet selv!
        tab.backward()
        optimizer.step()
    print(f"epoke {epoke + 1}: tab {tab.item():.4f}")

In [ ]:
with torch.no_grad():
    rekonstruktioner = model(X_test[:8])

vis_rekonstruktioner(X_test[:8], rekonstruktioner, n=8,
                     titel="784 pixels → 32 tal → 784 pixels")

Billederne har været igennem et nåleøje på 32 tal — en **kompression på 96 %** — og
er stadig tydeligt læsbare (lidt blødere i kanterne, detaljer er ofret). Netværket har
selv opfundet et "ciffer-sprog" med 32 ord.

## Hvor lille kan flaskehalsen blive?

Lad os presse citronen: samme netværk, fire flaskehals-størrelser:

In [ ]:
for flaskehals in [2, 8, 32, 64]:
    torch.manual_seed(42)
    m = Autoencoder(flaskehals=flaskehals)
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    for epoke in range(10):
        for i in range(0, len(X_traen), 64):
            X_batch = X_traen[i:i + 64]
            optimizer.zero_grad()
            tab = tabsfunktion(m(X_batch), X_batch)
            tab.backward()
            optimizer.step()
    with torch.no_grad():
        rek = m(X_test[:8])
    vis_rekonstruktioner(X_test[:8], rek, n=8, titel=f"flaskehals = {flaskehals} tal")

Ved 64 og 32: fine cifre. Ved 8: læsbart men udglattet. Ved **2**: netværket kan
kun huske "cirka hvilket ciffer" — alt andet er væk. Og netop 2-tals-udgaven gemmer på
notebookens flotteste figur...

## Det latente rum: netværkets mentale landkort

Med flaskehals = 2 kan hvert billede beskrives med to tal — altså et PUNKT i planen!
Lad os encode hele testsættet og farve punkterne efter, hvilket ciffer de er. Husk:
netværket har ALDRIG set labels:

In [ ]:
torch.manual_seed(42)
model2d = Autoencoder(flaskehals=2)
optimizer = torch.optim.Adam(model2d.parameters(), lr=0.001)
for epoke in range(10):
    for i in range(0, len(X_traen), 64):
        X_batch = X_traen[i:i + 64]
        optimizer.zero_grad()
        tab = tabsfunktion(model2d(X_batch), X_batch)
        tab.backward()
        optimizer.step()

with torch.no_grad():
    koder = model2d.encoder(X_test)

plt.figure(figsize=(8, 7))
spredningsplot = plt.scatter(koder[:, 0], koder[:, 1], c=y_test, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(spredningsplot, label="ciffer")
plt.title("Hele testsættet som punkter i det latente rum")
plt.show()

**Se!** Uden at kende ét eneste label har netværket sorteret cifrene i egne områder —
0'erne bor ét sted, 1'erne et andet... Kompression TVANG det til at forstå strukturen.
(Områderne overlapper hist og her — med kun 2 tal er der trængsel. Men at strukturen
overhovedet opstår af sig selv, er en af de smukkeste observationer i deep learning.)

### Opgaver

##### Opgave 10.1
Forklar med jeres egne ord: hvorfor lærer netværket ikke bare at "kopiere billedet
igennem"? Hvad er det præcis, flaskehalsen forhindrer — og hvad tvinger den i stedet
netværket til at gøre?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> En kopi kræver, at alle 784 pixelværdier kan komme igennem — men der er kun plads til 32 tal. Netværket SKAL altså smide information væk og kan kun få et godt tab ved at gemme det, der bedst beskriver billedet (form, hældning, tykkelse...) og genopbygge resten fra erfaring med, hvordan cifre plejer at se ud. Kompression = tvungen forståelse. (Samme grund til, at et godt resumé er sværere at skrive end en afskrift!)</span>

##### Opgave 10.2
Udfyld tabslinjen i træningsloopet — hvad skal rekonstruktionen sammenlignes med?

In [ ]:
torch.manual_seed(42)
m_test = Autoencoder(flaskehals=16)
optimizer = torch.optim.Adam(m_test.parameters(), lr=0.001)
for epoke in range(3):
    for i in range(0, len(X_traen), 64):
        X_batch = X_traen[i:i + 64]
        optimizer.zero_grad()
        tab = tabsfunktion(m_test(X_batch), X_batch)   # ← facit er inputtet selv
        tab.backward()
        optimizer.step()
print(f"tab: {tab.item():.4f}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>tabsfunktion(m_test(X_batch), X_batch)</code> — billedet er sit eget facit. Det er hele autoencoder-idéen i én linje, og grunden til at der ikke skal labels til.</span>

##### Opgave 10.3
Kig på de fire rekonstruktions-figurer (flaskehals 2/8/32/64). Beskriv præcis, HVAD
der forsvinder, når flaskehalsen strammes — er det tilfældige pixels, eller er det
bestemte TYPER af information? Og ved flaskehals 2: hvad gætter netværket åbenlyst
ud fra?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Det er ikke tilfældige pixels, der ryger — det er DETALJER: først fine kanter og småkrøller (64→32), så personlig håndskrift-stil (32→8), og til sidst alt undtagen "hvilket ciffer er det cirka" (8→2). Ved flaskehals 2 tegner decoderen reelt bare en generisk udgave af det ciffer, koden ligger nærmest — den gætter fra kategori, ikke fra billedet. Kompressionen smider det MINDST forudsigelige væk først — præcis som JPEG og MP3 gør med detaljer, dit øje/øre alligevel ikke savner.</span>

##### Opgave 10.4
Skil maskinen ad: encode ét billede og PRINT dets 32-tals kode — og decode den så
igen. Prøv at ændre ét af de 32 tal kraftigt (fx sæt `kode[0, 5] = 10.0`) før
decodningen. Hvad sker der med billedet?

In [ ]:
billede = X_test[0:1]

with torch.no_grad():
    kode = model.encoder(billede)
    print("billedet som 32 tal:")
    print(kode.numpy().round(2))

    kode[0, 5] = 10.0        # skru voldsomt op for én af de 32 "drejeknapper"
    genskabt = model.decoder(kode)

vis_rekonstruktioner(billede, genskabt, n=1)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Billedet ændrer sig — typisk deformeres cifret (bliver tykkere, hælder, glider mod et andet ciffer...), alt efter hvad netop dét tal koder for. De 32 tal er reelt 32 drejeknapper, netværket selv har opfundet ("hældning", "buethed",...). At udforske dem er at aflure, HVAD netværket har valgt at forstå — og præcis den idé (pil ved den latente kode) er kernen i moderne billedgeneratorer.</span>

##### Opgave 10.5 (find fejlen)
En kammerats autoencoder rekonstruerer nogle underligt "beskidte" billeder med grå
baggrund, og tabet sidder fast højere end jeres. Sammenlign kammeratens decoder med
den rigtige — hvad mangler der til sidst, og hvorfor betyder det noget, når pixels
ligger mellem 0 og 1?

In [ ]:
class KammeratAutoencoder(nn.Module):
    def __init__(self, flaskehals=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, flaskehals))
        self.decoder = nn.Sequential(
            nn.Linear(flaskehals, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid())    # ← sigmoiden manglede!

    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(42)
m_k = KammeratAutoencoder()
optimizer = torch.optim.Adam(m_k.parameters(), lr=0.001)
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        X_batch = X_traen[i:i + 64]
        optimizer.zero_grad()
        tab = tabsfunktion(m_k(X_batch), X_batch)
        tab.backward()
        optimizer.step()
print(f"tab: {tab.item():.4f}")
with torch.no_grad():
    vis_rekonstruktioner(X_test[:8], m_k(X_test[:8]), n=8)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Decoderen manglede sin afsluttende <code>nn.Sigmoid()</code>. Uden den kan outputtet være vilkårlige tal (også negative og over 1), mens målet ligger pænt i [0, 1] — netværket bruger så kræfter på at jagte intervallet i stedet for indholdet, og pixels uden for [0, 1] tegnes "beskidt". Reglen fra Intro-ML opgave 11.8 gælder stadig: match output-aktiveringen til, hvad outputtet SKAL være — også når outputtet er et billede.</span>

##### Opgave 10.6
Zoom ind i det latente rum: hvilke cifre overlapper mest i 2D-plottet? Tegn plottet
igen med KUN to udvalgte cifre (fx 4 og 9), og find et par, der er (næsten) perfekt
adskilt. Giver forvekslingerne mening rent visuelt?

In [ ]:
a, b = 4, 9      # ← prøv også (0, 1), (3, 8), (1, 7) ...
udvalg = (y_test == a) | (y_test == b)

plt.figure(figsize=(7, 6))
plt.scatter(koder[udvalg, 0], koder[udvalg, 1], c=y_test[udvalg], cmap="tab10", s=12)
plt.title(f"latent rum: kun {a} og {b}")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 4/9 og 3/8 (og 5/3) smelter typisk mest sammen — præcis de par, der også LIGNER hinanden i håndskrift (et lukket 4-tal er nærmest et 9-tal). 0/1 er derimod normalt fint adskilt: de er visuelt ekstremer. Autoencoderens landkort afspejler altså ægte visuel lighed — den har lært "ligner hinanden" uden nogensinde at have fået det at vide.</span>

##### Opgave 10.7 (ekstra)
**Morphing-maskinen:** tag to billeder (fx et 0 og et 1), encode dem til to punkter i
det latente rum — og decode så 10 punkter på LINJEN imellem dem. Udfyld
blandings-linjen og se det ene ciffer glide over i det andet.

In [ ]:
billede_a = X_test[y_test == 0][0:1]
billede_b = X_test[y_test == 1][0:1]

with torch.no_grad():
    kode_a = model2d.encoder(billede_a)
    kode_b = model2d.encoder(billede_b)

    mellembilleder = []
    for t in np.linspace(0, 1, 10):
        blandet_kode = (1 - t) * kode_a + t * kode_b
        mellembilleder.append(model2d.decoder(blandet_kode))

fig, akser = plt.subplots(1, 10, figsize=(15, 2))
for akse, billede in zip(akser, mellembilleder):
    akse.imshow(billede.reshape(28, 28), cmap="gray")
    akse.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>(1 - t) * kode_a + t * kode_b</code> — en ret linje i det latente rum. Undervejs opstår billeder, der ALDRIG har eksisteret: netværket digter meningsfulde mellemformer, fordi rummet mellem to ægte cifre også afkodes til noget ciffer-agtigt. Dét — at bevæge sig rundt i et lært latent rum og afkode — er i familie med idéen bag moderne generativ AI (diffusion, VAE'er m.m.). I har lige bygget en nano-udgave.</span>

##### Opgave 10.8
784 tal → 32 tal er ~96 % kompression, og cifrene overlever. Hvorfor kan denne
autoencoder så IKKE bare erstatte JPEG/zip til alle billeder? (Prøv evt. tanken: hvad
ville der ske, hvis I fodrede den med et foto af jeres kat?)

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Fordi den kun kan komprimere det, den er trænet på: håndskrevne cifre. Dens 32 tal er "ciffer-drejeknapper" — et kattefoto ville blive presset gennem ciffer-sproget og komme ud som... et udtværet ciffer-spøgelse (I tester præcis dét i anomali-afsnittet!). JPEG er generel (og garanterer en fejlgrænse); autoencoderen er hyper-specialiseret. Til gengæld slår den generelle metoder INDEN FOR sit speciale — det er hele emnets pointe, nu i kompressions-udgave.</span>

# 11: Støjfjernelse og alarmklokker

## Denoising: giv netværket et sværere job

Nu bliver det smart for alvor: giv netværket **støjede billeder som input**, men de
**RENE billeder som facit**. Så lærer det ikke bare at komprimere — det lærer at
**fjerne støj**. Og det kan det, fordi det ved, hvordan cifre SKAL se ud (støj passer
ikke ind i ciffer-sproget og bliver simpelthen ikke gen-tegnet):

In [ ]:
def tilfoej_stoej(X, styrke=0.3):
    stoejet = X + styrke * torch.randn(X.shape)
    return stoejet.clamp(0, 1)              # pixels skal blive i [0, 1]

torch.manual_seed(42)
denoiser = Autoencoder(flaskehals=32)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=0.001)

for epoke in range(10):
    for i in range(0, len(X_traen), 64):
        X_ren = X_traen[i:i + 64]
        X_stoejet = tilfoej_stoej(X_ren)
        optimizer.zero_grad()
        tab = tabsfunktion(denoiser(X_stoejet), X_ren)    # støjet ind → RENT facit!
        tab.backward()
        optimizer.step()
    print(f"epoke {epoke + 1}: tab {tab.item():.4f}")

In [ ]:
torch.manual_seed(7)
X_test_stoejet = tilfoej_stoej(X_test[:8])

with torch.no_grad():
    renset = denoiser(X_test_stoejet)

vis_rekonstruktioner(X_test_stoejet, renset, n=8,
                     titel="øverst: hvad netværket FÅR — nederst: hvad det leverer")

Fra sne-storm til læsbare cifre. Præcis samme princip bruges i virkeligheden til at
rense skanninger, gamle fotos og astronomi-billeder — og tankegangen "lær at fjerne
støj" er tilmed kernen i diffusions-modeller (dem bag AI-billedgeneratorer!).

## Anomali-detektion: alarmen der ikke skal have eksempler

Sidste superkraft. Autoencoderen er EKSPERT i cifre — og netop derfor **elendig** til
alt andet. Fodrer man den med noget, der ikke ligner træningsdataene, bliver
rekonstruktionen dårlig, og **rekonstruktionsfejlen eksploderer**. Det kan bruges som
alarm: høj fejl = "det her har jeg aldrig set før!"

Det geniale: man behøver INGEN eksempler på det unormale for at træne alarmen — kun
masser af normalt. (Tænk kreditkort: millioner af normale køb, næsten ingen kendte
svindler.) Vi tester med "unormale" billeder, vi nemt kan lave: inverterede cifre
(hvid baggrund, sort skrift) og ren støj:

In [ ]:
def rekonstruktionsfejl(model, X):
    with torch.no_grad():
        rek = model(X)
    return ((rek - X) ** 2).mean(dim=1)          # én fejl PR. BILLEDE

normale = X_test[:500]
inverterede = 1.0 - X_test[500:1000]             # negativ-udgaver
torch.manual_seed(1)
stoejbilleder = torch.rand(500, 784)             # ren tilfældighed

fejl_normale = rekonstruktionsfejl(model, normale)
fejl_inverterede = rekonstruktionsfejl(model, inverterede)
fejl_stoej = rekonstruktionsfejl(model, stoejbilleder)

plt.hist(fejl_normale, bins=40, alpha=0.6, label="normale cifre")
plt.hist(fejl_inverterede, bins=40, alpha=0.6, label="inverterede cifre")
plt.hist(fejl_stoej, bins=40, alpha=0.6, label="ren støj")
plt.xlabel("rekonstruktionsfejl")
plt.ylabel("antal billeder")
plt.legend()
plt.title("Alarmen: unormale input får høj fejl")
plt.show()

Tre pænt adskilte pukler! Sæt en grænse mellem dem, og I har en vagthund, der aldrig
har set en indbrudstyv, men ved præcis, hvordan "normalt" ser ud.

### Opgaver

##### Opgave 11.1
Skru på støjstyrken: test denoiseren (trænet på styrke 0.3) med støj på **0.1, 0.5 og
1.0**. Hvornår giver den op — og HVORDAN giver den op? (Gætter den forkert, eller
tegner den bare grød?)

In [ ]:
for styrke in [0.1, 0.5, 1.0]:
    torch.manual_seed(7)
    X_s = tilfoej_stoej(X_test[:8], styrke=styrke)
    with torch.no_grad():
        renset = denoiser(X_s)
    vis_rekonstruktioner(X_s, renset, n=8, titel=f"støjstyrke {styrke}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 0.1: næsten perfekt rensning. 0.5: stadig imponerende (den er trænet på 0.3 men generaliserer opad). 1.0: signalet drukner — og bemærk HVORDAN den fejler: den tegner stadig pæne, ciffer-agtige former, bare tit det FORKERTE ciffer. Den hallucinerer det mest plausible ciffer ud fra resterne — den slags "selvsikre fejl" er både modellens styrke og dens farligste egenskab.</span>

##### Opgave 11.2
Udfyld støj-funktionen: læg gaussisk støj til og "clamp" resultatet, så pixelværdierne
bliver i [0, 1].

In [ ]:
def min_stoej(X, styrke=0.3):
    stoejet = X + styrke * torch.randn(X.shape)
    return stoejet.clamp(0, 1)

vis_billeder(min_stoej(X_test[:5]), y_test[:5], n=5)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>styrke * torch.randn(X.shape)</code> lægger normalfordelt støj til hver pixel, og <code>.clamp(0, 1)</code> klipper af, så vi ikke får "umulige" pixels. (<code>randn</code> = normalfordelt, <code>rand</code> = jævnt fordelt — et bogstav til forskel, klassisk fælde.)</span>

##### Opgave 11.3
Sammenlign den ALMINDELIGE autoencoder (`model`) med `denoiser`-modellen på de samme
støjede billeder. Hvem renser bedst — og hvorfor giver det mening, når de har set
forskellige ting under træningen?

In [ ]:
torch.manual_seed(7)
X_s = tilfoej_stoej(X_test[:8])

with torch.no_grad():
    vis_rekonstruktioner(X_s, model(X_s), n=8, titel="almindelig autoencoder")
    vis_rekonstruktioner(X_s, denoiser(X_s), n=8, titel="denoiser")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Denoiseren vinder klart: den almindelige autoencoder gengiver en del af støjen (småpletter og grums følger med igennem flaskehalsen), for den er trænet til at REPRODUCERE sit input så præcist som muligt — også dets fejl. Denoiseren er derimod trænet til at oversætte støjet → rent, så den har lært aktivt at IGNORERE støj. Træningsopgaven definerer, hvad modellen bliver god til — ned til mindste detalje.</span>

##### Opgave 11.4 (find fejlen)
Denne denoiser-træning ser rigtig ud... men modellen lærer bare at beholde støjen. Kig
GODT på tabslinjen: hvad sammenlignes rekonstruktionen med — og hvad SKULLE den
sammenlignes med?

In [ ]:
torch.manual_seed(42)
d_fejl = Autoencoder(flaskehals=32)
optimizer = torch.optim.Adam(d_fejl.parameters(), lr=0.001)
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        X_ren = X_traen[i:i + 64]
        X_stoejet = tilfoej_stoej(X_ren)
        optimizer.zero_grad()
        tab = tabsfunktion(d_fejl(X_stoejet), X_ren)    # ← facit skal være det RENE billede!
        tab.backward()
        optimizer.step()

torch.manual_seed(7)
X_s = tilfoej_stoej(X_test[:8])
with torch.no_grad():
    vis_rekonstruktioner(X_s, d_fejl(X_s), n=8, titel="nu renses støjen")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Tabet sammenlignede med <code>X_stoejet</code> — det støjede input selv. Så var opgaven "genskab støjen præcist", og det gjorde modellen lydigt. Med <code>X_ren</code> som facit bliver opgaven "se igennem støjen". Modellen gør ALDRIG, hvad du mener — kun hvad tabsfunktionen siger. Den sætning er værd at tage med hjem fra hele campen.</span>

##### Opgave 11.5
Udfyld fejl-pr.-billede-beregningen (kvadrér forskellen og tag gennemsnittet — men
KUN over pixel-dimensionen, så hvert billede får sin egen fejl).

In [ ]:
def min_rekonstruktionsfejl(model, X):
    with torch.no_grad():
        rek = model(X)
    return ((rek - X) ** 2).mean(dim=1)

fejl = min_rekonstruktionsfejl(model, X_test[:10])
print(fejl)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>mean(dim=1)</code> — gennemsnit hen over de 784 pixels, så der bliver ét tal pr. billede (shape (10,)). Uden <code>dim</code> får man ét samlet tal for hele bunken, og så kan man ikke pege på DET mistænkelige billede. dim-disciplin, igen!</span>

##### Opgave 11.6
Byg selv alarmen færdig: vælg en alarm-grænse ud fra histogrammet, og beregn (a) hvor
stor en andel af de NORMALE cifre der fejlagtigt udløser alarm, og (b) hvor stor en
andel af de INVERTEREDE der bliver fanget. Flyt grænsen og se de to tal trække i hver
sin retning.

In [ ]:
for graense in [0.02, 0.03, 0.05, 0.08]:
    falske_alarmer = (fejl_normale > graense).float().mean()
    fangede = (fejl_inverterede > graense).float().mean()
    print(f"grænse {graense}: falske alarmer {falske_alarmer.item():.1%}, fangede {fangede.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Klassisk skydeskive: lav grænse fanger alt unormalt, men råber også ad normale billeder; høj grænse er tavs og misser. På vores data findes der typisk en grænse, der fanger næsten 100 % af de inverterede med få procent falske alarmer — men i virkeligheden (svindel, kræftscreening...) er afvejningen benhård og bør, ligesom i hjerte-notebooken, ikke træffes af programmøren alene.</span>

##### Opgave 11.7 (ekstra)
Hvilket CIFFER er sværest at rekonstruere? Beregn den gennemsnitlige
rekonstruktionsfejl pr. ciffer-klasse (for-løkke over 0-9) og plot som søjler. Har I
en teori om, hvorfor netop dét ciffer er svært?

In [ ]:
fejl_alle = rekonstruktionsfejl(model, X_test)

gennemsnit_pr_ciffer = []
for ciffer in range(10):
    gennemsnit_pr_ciffer.append(fejl_alle[y_test == ciffer].mean().item())

plt.bar(range(10), gennemsnit_pr_ciffer)
plt.xlabel("ciffer")
plt.ylabel("gns. rekonstruktionsfejl")
plt.xticks(range(10))
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk topper 8, 5 og 2, mens 1 er suverænt lettest. Teorien holder vand: et 1-tal er en streg (næsten ingen information at gemme), mens 8-taller og 5-taller har mange kurver, sløjfer og skrivestile — mere at komprimere på for få pladser i flaskehalsen. Rekonstruktionsfejlen er reelt et mål for, hvor KOMPLEKS og VARIERET en klasse er.</span>

##### Opgave 11.8
Nævn to virkelige systemer, hvor "træn kun på det normale, og slå alarm ved høj
rekonstruktionsfejl" ville være oplagt — og beskriv for mindst ét af dem, hvad
metodens svaghed ville være i praksis. (Hint til svagheden: hvad sker der, når
"normalt" ændrer sig med tiden?)

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Oplagte eksempler: kreditkortsvindel (millioner af normale køb, få kendte svindler), fejl på fabriksdele/vindmøller (lyt/kig efter det, der ikke plejer at ske), netværksangreb, unormale hjerterytmer, kvalitetskontrol af fødevarer. Svagheden: "normalt" er en bevægelig størrelse — nye købsvaner (Black Friday!), nyt maskinslid, nye årstider... Alarmen råber så ad harmløse nyheder, indtil man gentræner. Anomali-detektion kræver vedligehold — en model er ikke et produkt, man bygger én gang, men noget man PASSER.</span>

# Godt gået — og det var hele emnet!

Autoencoderen viste jer unsupervised deep learning: kompression som tvungen forståelse,
det latente rum (med morphing!), støjfjernelse og anomali-alarmer — alt sammen uden ét
eneste label.

**Den færdige værktøjskasse:**

| Datatype / opgave | Værktøj |
|---|---|
| tabeldata | beslutningstræer & random forests |
| små/støjede datasæt, al NN-træning | validering, early stopping, dropout, weight decay |
| billeder | CNN |
| grupper uden labels | k-means clustering |
| sekvenser & tid | RNN/LSTM |
| kompression, støj, anomalier | autoencodere |
| sprog | transformere — **campens finale!** |

Det rigtige værktøj til det rigtige job. God fornøjelse med finalen!